# 03 - YOLO detection + a VLM: trigger-based multimodal

**Why put a detector in front of a VLM at all?** Because the two models live at opposite
ends of the cost spectrum:

| | YOLO11 detector | Vision-Language Model (Qwen3-VL-4B) |
| --- | --- | --- |
| Cost | **cheap** (tens of fps) | **expensive** (seconds per answer) |
| Cadence | **every frame**, always-on | **only on interesting events** |
| Output | boxes: *where* and *what class* | language: *what is happening* |

The detector is the **always-on trigger**; the VLM is the **expensive specialist** you call
sparingly. This notebook explains the design of `apps/detection-vlm-assistant/`: how to decide
which detections deserve a VLM call, how to design the prompt, direct model vs server, and why
the VLM can never run per frame.


## The core arithmetic: why never per-frame

A 30 fps stream delivers a frame every **33 ms**. A 4B VLM answer takes on the order of
**1-3 seconds**. If you called the VLM inline, per frame, you would block the detection loop
for ~30-90 frames per call - the "always-on" detector would stall and the stream would back up.

So the rule is structural, not an optimization:

> The detector runs on **every** frame. The VLM runs on a **tiny, gated, de-duplicated**
> subset of frames, from a **separate thread**, behind a **bounded queue** that drops work
> when the VLM is busy.

The detection loop pushes at most one crop per interval into the queue and immediately moves
on; a background worker pulls crops and calls the VLM. If the worker is busy, new triggers are
dropped (back-pressure), never queued unboundedly. That is how a 1-3 s model coexists with a
tens-of-fps loop.

## Which detections deserve a VLM call? (crop selection)

Most detections are not worth a VLM call. Four filters, cheapest first, decide what is:

1. **Class allow-list** - e.g. only `person`. A VLM caption of the 900th detected `car` is noise.
2. **Confidence gate** - skip low-score boxes; a bad crop yields a bad caption.
3. **Area gate** - skip tiny/far objects (a 12x8 px crop tells the VLM nothing). Require the
   crop to cover at least a few percent of the frame.
4. **De-duplication** - the *same* person standing still is detected every frame. Without dedup
   you would fire the VLM ~30x/second on one object. Track recently-fired boxes and suppress a
   new trigger whose IoU with a recent one is high, until a cooldown elapses.

Then a **wall-clock rate limit** caps the global VLM call rate regardless of scene activity.

Here is the gating logic from `src/vlm_commenter.py` (illustrative; not executed here):

In [1]:
# Illustrative - the gating + dedup from apps/detection-vlm-assistant/src/vlm_commenter.py.
def passes_gate(box, frame_area, cfg, label_fn):
    if box["score"] < cfg.vlm_trigger_min_score:        # (2) confidence
        return False
    label = label_fn(box["class_id"]).lower()
    if cfg.vlm_trigger_classes and label not in cfg.vlm_trigger_classes:   # (1) class
        return False
    area = (box["x2"] - box["x1"]) * (box["y2"] - box["y1"])
    if frame_area > 0 and area / frame_area < cfg.vlm_trigger_min_area_frac:  # (3) area
        return False
    return True

def iou(a, b):
    ix1, iy1 = max(a["x1"], b["x1"]), max(a["y1"], b["y1"])
    ix2, iy2 = min(a["x2"], b["x2"]), min(a["y2"], b["y2"])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    if inter <= 0:
        return 0.0
    area_a = (a["x2"] - a["x1"]) * (a["y2"] - a["y1"])
    area_b = (b["x2"] - b["x1"]) * (b["y2"] - b["y1"])
    return inter / (area_a + area_b - inter)

# Dedup: a box is "the same object" as a recent trigger if same class AND iou >= dedup_iou,
# until dedup_cooldown_s elapses. select() then returns the highest-score NON-duplicate box,
# at most one per frame. Rate limit (vlm_interval_seconds) caps how often we enqueue at all.

**Interpretation.** The filters are ordered cheapest-first: a scalar compare, a set
lookup, one multiply, then IoU only against a short list of recent triggers. The whole gate
costs microseconds - negligible against the detector step (tens of ms), so it is safe to run
every frame. Tune the four knobs in `config/default.conf`:

- `vlm_trigger_classes=person`
- `vlm_trigger_min_score=0.55`
- `vlm_trigger_min_area_frac=0.02`
- `vlm_dedup_iou=0.6`, `vlm_dedup_cooldown_s=15.0`, `vlm_interval_seconds=5.0`

## Colour correctness - the one bug that silently ruins captions

VLM image inputs must be **`uint8` HWC RGB**. OpenCV decodes and crops in **BGR**. If you hand
a BGR crop to the request, the red and blue channels are swapped: the model still answers, but
about a wrongly-coloured image - a red coat becomes blue, skin tones go off, and the caption
degrades with no error anywhere.

So convert **at the request boundary**, right before building the `GenerationRequest`:

```python
crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)   # uint8 HWC RGB
```

Source: `GenAITypes.h` ("VLM requests require uint8 HWC RGB tensors") and `02-run-llm-vlm`.
The app keeps everything BGR (OpenCV-native) through detection and cropping and does this one
conversion in `src/vlm_commenter.py` at the boundary - the single place it matters.

## Prompt design for describing a crop

The crop is already the *what* and *where* (the detector gave you the class and box). The VLM's
job is the *what's happening / what's notable*. A good crop prompt:

- **States the context** - "this is a crop of one detected `{label}` from a security camera".
  The VLM sees a tight crop with no scene; telling it what it is looking at anchors the answer.
- **Asks one specific question** - "describe what this `{label}` is doing and anything notable".
- **Bounds the length** - one short sentence, plus `max_new_tokens` as a hard cap. Captions feed
  a log or an alert; long essays waste latency you cannot afford.
- **Substitutes the class** - `{label}` is filled from the detection, so the same template works
  for `person`, `car`, `dog`, etc.

The app's default template:

In [2]:
# apps/detection-vlm-assistant default prompt (config: vlm_prompt), {label} from the detection:
VLM_PROMPT = (
    "You are watching a security camera. This image is a crop of one detected {label}. "
    "Describe what this {label} is doing and anything notable in one short sentence."
)
# prompt = VLM_PROMPT.format(label="person")
# For alerting you might instead ask a yes/no: "Is this person holding a weapon? Answer yes or no."
# A narrower question -> a shorter, more reliable answer and lower latency.

**Interpretation.** Keep the prompt in config, not code, so operators can retune tone and
task (caption vs. classify vs. yes/no alert) without editing the app. Pair a narrow prompt with
a small `max_new_tokens` - the two together are your main latency control on the VLM side.

## Direct `VisionLanguageModel` vs `GenAIServer`

Two ways to reach the VLM:

| | Direct `pyneat.genai.VisionLanguageModel` | `GenAIServer` (OpenAI-compatible HTTP) |
| --- | --- | --- |
| Boundary | in-process function call | network (HTTP/JSON, base64 images) |
| Latency | lowest - no serialization, no hop | +encode/serialize/parse per call |
| When | one app owns detector **and** VLM (this app) | UI / separate service / remote client that must not link Neat |
| Trade | tightest coupling, one process holds both models | clean decoupling, language-agnostic clients, extra ops |

This app uses the **direct** path: the detection loop and the VLM live in one process, so a
function call is strictly better than looping back through HTTP. The upstream
`apps/examples/genai/detection-to-vlm-assistant` uses the **server** path (it posts base64 crops
to an OpenAI-compatible endpoint) - appropriate when the captioner is a shared, separate service.
`GenAIServer` itself is covered in `llima/05-genai-server`.

## The VLM call

The whole VLM leg of the app is `src/vlm_commenter.py`. Below is its essence: lazily load one
`VisionLanguageModel`, convert the crop to RGB, and run a `GenerationRequest`. This runs in a
**background worker**, not the detection loop.

> This cell loads a multi-GB VLM and runs inference. Run it on the DevKit through the app, as
> shown in **Run it** below, rather than here.

In [4]:
# Essence of apps/detection-vlm-assistant/src/vlm_commenter.py (loads a VLM; run on the DevKit).
import cv2
import numpy as np
import pyneat as neat

model = neat.genai.VisionLanguageModel(
    "/media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4")   # LM weights + vision encoder
print("model_id     :", model.model_id())
print("accepts_image:", model.accepts_image())   # must be True for a VLM directory

frame_bgr = cv2.imread("../../tutorial/assets/images/image.png")
y1=100
y2=200
x1= 220
x2 = 320
crop_bgr = frame_bgr[y1:y2, x1:x2]  # OpenCV-native BGR crop from a detection box
# CRITICAL: VLM images are uint8 HWC RGB. Convert at the request boundary.
crop_rgb = np.ascontiguousarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))

request = neat.genai.GenerationRequest()
request.prompt = VLM_PROMPT.format(label="person")
request.images = [crop_rgb]            # list of uint8 HWC RGB arrays
request.max_new_tokens = 96

result = model.run(request)
print(result.text)                     # the caption
m = result.metrics                     # generated_tokens, time_to_first_token_s, tokens_per_second
print(m.generated_tokens, "tok,", round(m.tokens_per_second, 1), "tok/s, ttft", m.time_to_first_token_s, "s")

model_id     : Qwen3-VL-4B-Instruct-GPTQ-a16w4
accepts_image: True
Loading model /media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4/elf_files/models--Qwen--Qwen3-VL-4B-Instruct_vision_stage1_mla.elf


[mlatiming] mlashm_load_model /media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4/elf_files/models--Qwen--Qwen3-VL-4B-Instruct_vision_stage1_mla.elf took 2572 ms ok=1


Done loading /media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4/elf_files/models--Qwen--Qwen3-VL-4B-Instruct_vision_stage1_mla.elf
Loading model /media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4/elf_files/models--Qwen--Qwen3-VL-4B-Instruct_GPTQ_language_n128_pre_layer0_stage1_mla.elf
Done loading /media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4/elf_files/models--Qwen--Qwen3-VL-4B-Instruct_GPTQ_language_n128_pre_layer0_stage1_mla.elf
Loading model /media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4/elf_files/models--Qwen--Qwen3-VL-4B-Instruct_GPTQ_language_n128_cache_token0_stage1_mla.elf
Done loading /media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4/elf_files/models--Qwen--Qwen3-VL-4B-Instruct_GPTQ_language_n128_cache_token0_stage1_mla.elf
Loading model /media/nvme/llima/models/Qwen3-VL-4B-Instruct-GPTQ-a16w4/elf_files/models--Qwen--Qwen3-VL-4B-Instruct_GPTQ_language_n128_post_layer0_stage1_mla.elf
Done loading /media/nvme/llima/models/Qwen3-VL-4B-Instr

**Interpretation.** `run(request)` = one encode + one generate and returns a
`GenerationResult` with `.text` and `.metrics`. `request.images` takes a list, so multiple crops
per call are possible. The **worker is bounded**: while `run()` blocks for seconds, the detection
loop keeps enqueuing at most `vlm_max_pending` crops and drops the rest, so latency is absorbed,
not propagated. If you will ask several questions about the *same* crop, `model.encode([rgb])`
once and set `request.use_cached_images = True` (see `02-run-llm-vlm`); for a stream of
*different* crops, caching buys nothing and the direct-image path is correct.

## Performance: how the queue absorbs VLM latency

The two models sit orders of magnitude apart:

- **Detector:** tens of fps; one detection step is on the order of tens of milliseconds.
- **VLM:** ~1-3 s per answer for a 4B VLM such as Qwen3-VL-4B.

That is a **~100x** cost ratio. The design consequences:

- `vlm_interval_seconds` (default 5 s) caps the VLM to at most one call per 5 s regardless of
  how busy the scene is - so the VLM handles ~0.2 calls/s while the detector handles tens of
  frames/s.
- `vlm_max_pending` (default 1) bounds queued/in-flight crops. When the worker is mid-call, new
  triggers are **dropped**, not queued - the detection loop never blocks and memory stays bounded.
- **Dedup** stops the same stationary object from consuming every VLM slot.

Net effect: the detector stays real-time; the VLM annotates a trickle of the most interesting,
distinct events. Raising the VLM rate does not speed the VLM up - it just starves the detector.
The correct knob for "more captions" is a *smaller, cheaper VLM* or a *narrower prompt*, not a
shorter interval.

## Run it (on the DevKit)

Both the detection leg and the VLM leg go through the same app. Run from the app directory:

```bash
cd apps/detection-vlm-assistant
```

**Detection leg + dry-run VLM** (logs the crop and the exact prompt that *would* be sent; never
loads or calls the VLM - this is how you exercise the detection leg without a VLM):

```bash
dk ./main.py --no-vlm --frames 40
```

**Still-image dry-run** over a folder (deterministic, no RTSP server needed):

```bash
dk ./main.py --source image --image ../../model-compilation/assets/yolo_calibration --no-vlm
```

**Full pipeline with the real VLM** (run once the VLM directory is present):

```bash
dk ./main.py --config ./config/default.conf
```

## Expected output

**Detection leg:** per-frame detection counts, and a dry-run block each time a person clears the
gate:

```text
detector=yolo_11n_mpk.tar.gz decode=yolo11 vlm=DRY-RUN (crop+prompt logged, VLM not called)
rtsp=rtsp://<rtsp-host>:8555/stream stream=1280x720@60
frame=1 detections=13 fps=14.57
vlm[dry-run] WOULD send crop -> VLM
  class   : PERSON score=0.81
  bbox    : (927, 429, 1019, 666)
  crop    : shape=(237, 92, 3) (BGR; converted to RGB before the request)
  prompt  : 'You are watching a security camera. This image is a crop of one detected person. ...'
frame=30 detections=12 fps=53.65
```

Still-image mode prints real classes, e.g.
`000000000885.jpg detections=4 [PERSON:0.92, PERSON:0.85, PERSON:0.66, TENNIS RACKET:0.81]`.

**VLM leg:** each dry-run block is replaced by a caption line:

```text
vlm[PERSON score=0.81 bbox=(927, 429, 1019, 666)]: A person in a dark jacket is walking across
the platform carrying a bag.  (58 tok, 22.4 tok/s, ttft=0.34s)
```

Exact wording depends on the crop and the model's sampling; the shape of the output is what to
verify.

## References

- App: `apps/detection-vlm-assistant/` (`main.py`, `src/vlm_commenter.py`, `README.md`)
- `apps/examples/genai/detection-to-vlm-assistant` (upstream crop-to-VLM, server path)
- `apps/multi-stream-yolo-yolo11/main.py` (YOLO11 detector, `push`+`pull`, `decode_bbox`)
- [`VisionLanguageModel.h`](https://github.com/sima-neat/core/blob/main/include/genai/VisionLanguageModel.h), [`GenAITypes.h`](https://github.com/sima-neat/core/blob/main/include/genai/GenAITypes.h)
- [`module.cpp`](https://github.com/sima-neat/core/blob/main/python/src/module.cpp) — `pyneat.genai` bindings, `decode_bbox`
- `../02-run-llm-vlm/02_run_vlm.ipynb` (VLM API + image-format detail)